# **Instructions**

This notebook converts a Jupyter notebook stored in Google Drive to **HTML** or **PDF** format using `nbconvert`.

Run each cell **in order** from top to bottom:
1. **Install Latex tools** – installs the packages needed for PDF conversion (only required once per session).
2. **Mount Google Drive** – connects your Google Drive so the notebook can read and write files.
3. **Select your notebook** – use the two dropdowns to pick the folder and `.ipynb` file you want to convert.
4. **Output as HTML** – converts the notebook to an HTML file (useful for printing in landscape orientation from a browser).
5. **Output to PDF** – converts the notebook directly to a PDF file in portrait orientation.

# **Install Latex tools**

Installs the LaTeX packages (`texlive`, `texlive-xetex`, `texlive-latex-extra`, and `pandoc`) required to convert notebooks to PDF. This only needs to be run **once per session** — if you restart the runtime, run this cell again before converting.

In [ ]:
# Refresh the list of available packages. Sometimes Colab container doesn't have all packages at startup.
print("==== Updating package lists... ====")
!apt-get update -qq

# Install packages. Silence is a virtue.
print("\n==== Installing LaTeX tools (this may take some time)... ====")
!apt-get install -y texlive texlive-xetex texlive-latex-extra pandoc 1>/dev/null

# **Mount Google Drive**

Mounts your Google Drive at `/content/drive`, making all your Drive files accessible from this notebook. You will be prompted to grant access if this is your first time. The drive must be mounted before you can select or convert any notebook.

In [ ]:
import os
from google.colab import drive

if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')

# **Select Your Notebook**

After mounting your Drive, use the two dropdowns below to locate your notebook:
1. **Folder** – pick the folder in your Google Drive that contains the notebook.
2. **Notebook** – pick the `.ipynb` file within that folder.

Selecting a file automatically sets the `WORKING_DIR` and `HW_FILE` variables used by the conversion cells below.

In [ ]:
import os
import ipywidgets as widgets
from IPython.display import display, clear_output

# Walk Google Drive and build a dict: folder -> [list of .ipynb filenames]
drive_root = '/content/drive/MyDrive'
folder_map = {}
for dirpath, dirnames, filenames in os.walk(drive_root):
    notebooks = [f for f in filenames if f.endswith('.ipynb')]
    if notebooks:
        # Use relative folder path from MyDrive (or '.' for root)
        rel_dir = os.path.relpath(dirpath, drive_root)
        folder_map[rel_dir] = sorted(notebooks)

sorted_folders = sorted(folder_map.keys())

# Folder dropdown
folder_picker = widgets.Dropdown(
    options=sorted_folders,
    description='Folder:',
    layout=widgets.Layout(width='600px'),
    style={'description_width': 'initial'}
)

# Notebook dropdown (populated from the selected folder)
initial_folder = sorted_folders[0] if sorted_folders else ''
file_picker = widgets.Dropdown(
    options=folder_map.get(initial_folder, []),
    description='Notebook:',
    layout=widgets.Layout(width='600px'),
    style={'description_width': 'initial'}
)

status_out = widgets.Output()

def set_selection(folder, notebook):
    """Update the global WORKING_DIR / HW_FILE variables."""
    global WORKING_DIR, HW_FILE
    WORKING_DIR = '' if folder == '.' else folder
    HW_FILE = notebook
    with status_out:
        clear_output()
        print(f"  WORKING_DIR = '{WORKING_DIR}'")
        print(f"  HW_FILE     = '{HW_FILE}'")

def on_folder_changed(change):
    """Repopulate the notebook dropdown when the folder changes."""
    new_folder = change['new']
    file_picker.options = folder_map.get(new_folder, [])
    # on_file_changed will fire automatically when options reset

def on_file_changed(change):
    set_selection(folder_picker.value, change['new'])

folder_picker.observe(on_folder_changed, names='value')
file_picker.observe(on_file_changed, names='value')

# Initialize with the current selection
set_selection(initial_folder, file_picker.value if file_picker.options else '')

print("Select a folder:")
display(folder_picker)

print("\nSelect a notebook:")
display(file_picker)

print("\nCurrent selection:")
display(status_out)

# **Output as HTML**

Converts the selected notebook to an **HTML** file saved alongside the original in your Google Drive. HTML is ideal for printing in **landscape orientation** — open the file in a browser and use the browser's Print function to save as PDF with custom margins and orientation.

In [ ]:
# Convert the notebook to HTML (useful for landscape printing from a browser)
!jupyter nbconvert --to html --output "/content/drive/MyDrive/{WORKING_DIR}/{HW_FILE}.html" "/content/drive/MyDrive/{WORKING_DIR}/{HW_FILE}"

# **Output to PDF**

Converts the selected notebook directly to a **PDF** file in **portrait orientation** using LaTeX (`xelatex`) and saves it alongside the original in your Google Drive. Use this for standard portrait-format printouts. Note: this requires the LaTeX tools installed in the first cell.

In [ ]:
# Convert the notebook to PDF in portrait orientation
!jupyter nbconvert --to pdf --output "/content/drive/MyDrive/{WORKING_DIR}/{HW_FILE}.pdf" "/content/drive/MyDrive/{WORKING_DIR}/{HW_FILE}"